In [5]:
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")

print("Token loaded:", token is not None)


Token loaded: True


In [6]:
import os

os.environ["GITHUB_TOKEN"] = token

!git clone https://$GITHUB_TOKEN@github.com/nehnamehranmk638-dev/multilingual-rag-research.git

Cloning into 'multilingual-rag-research'...
remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 46 (delta 16), reused 28 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (46/46), 306.05 KiB | 19.13 MiB/s, done.
Resolving deltas: 100% (16/16), done.


In [7]:
%cd /content/multilingual-rag-research

/content/multilingual-rag-research


In [8]:
!git config --global credential.helper store

In [9]:
import subprocess

username = "nehnamehrankmk638-dev"

credential = f"""protocol=https
host=github.com
username={username}
password={token}

"""

subprocess.run(
    ["git", "credential", "approve"],
    input=credential,
    text=True,
    check=True
)

print("GitHub authentication configured.")



GitHub authentication configured.


In [10]:
!git fetch origin
!git switch nehna


branch 'nehna' set up to track 'origin/nehna'.
Switched to a new branch 'nehna'


In [11]:
!git pull

Already up to date.


In [12]:
!git status

On branch nehna
Your branch is up to date with 'origin/nehna'.

nothing to commit, working tree clean


In [13]:
!pip install -q sentence-transformers faiss-cpu

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-base")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [3]:
def embed_passages(texts):
    prefixed = ["passage: " + t for t in texts]
    return model.encode(
        prefixed,
        normalize_embeddings=True,
        show_progress_bar=True
    )

def embed_queries(texts):
    prefixed = ["query: " + t for t in texts]
    return model.encode(
        prefixed,
        normalize_embeddings=True,
        show_progress_bar=True
    )

In [14]:
import json

with open("data/corpus.json", encoding="utf-8") as f:
    corpus = json.load(f)

with open("data/questions.json", encoding="utf-8") as f:
    questions = json.load(f)

corpus_texts = [doc["text"] for doc in corpus]
question_texts = [q["question"] for q in questions]

passage_embeddings = embed_passages(corpus_texts)
query_embeddings = embed_queries(question_texts)

print(passage_embeddings.shape)   # (1000, 768)
print(query_embeddings.shape)     # (100, 768)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

(1000, 768)
(100, 768)


In [15]:
corpus_texts = [doc["text"] for doc in corpus]
question_texts = [q["question"] for q in questions]

In [16]:
passage_embeddings = embed_passages(corpus_texts)
query_embeddings = embed_queries(question_texts)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

In [17]:
print("Passage embeddings:", passage_embeddings.shape)
print("Query embeddings:", query_embeddings.shape)

Passage embeddings: (1000, 768)
Query embeddings: (100, 768)


In [18]:
import numpy as np

def dense_search_numpy(query_vec, k=5):
    scores = passage_embeddings @ query_vec
    ranked_ids = np.argsort(-scores)
    return ranked_ids[:k].tolist()

In [19]:
q_idx = 0

print("Question:", questions[q_idx]["question"])
print("Gold passage id:", questions[q_idx]["gold_passage_id"])

top5 = dense_search_numpy(query_embeddings[q_idx], k=5)

print("Top 5 retrieved:", top5)

Question: What type of surnames is their a strong presence of?
Gold passage id: 956
Top 5 retrieved: [956, 125, 551, 569, 957]


In [20]:
print(corpus[top5[0]]["text"])

According to the same statistics, the average age of people living in Newcastle is 37.8 (the national average being 38.6). Many people in the city have Scottish or Irish ancestors. There is a strong presence of Border Reiver surnames, such as Armstrong, Charlton, Elliot, Johnstone, Kerr, Hall, Nixon, Little and Robson. There are also small but significant Chinese, Jewish and Eastern European (Polish, Czech Roma) populations. There are also estimated to be between 500 and 2,000 Bolivians in Newcastle, forming up to 1% of the population—the largest such percentage of any UK city.


In [21]:
dense_results_numpy = {}

for i, q in enumerate(questions):
    dense_results_numpy[q["question_id"]] = dense_search_numpy(
        query_embeddings[i],
        k=10
    )

In [22]:
import faiss

dim = passage_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(passage_embeddings)

def dense_search_faiss(query_vec, k=5):
    query_vec = query_vec.reshape(1, -1)
    scores, ids = index.search(query_vec, k)
    return ids[0].tolist()

# sanity check against the NumPy version
print(dense_search_faiss(query_embeddings[0], k=5))
print(dense_search_numpy(query_embeddings[0], k=5))

[956, 125, 551, 569, 957]
[956, 125, 551, 569, 957]


In [23]:
dense_results = {}

for i, q in enumerate(questions):
    dense_results[q["question_id"]] = dense_search_faiss(
        query_embeddings[i],
        k=10
    )

In [24]:
def recall_at_k(questions, all_results, k):
    hits = 0

    for q in questions:
        retrieved = all_results[q["question_id"]][:k]

        if q["gold_passage_id"] in retrieved:
            hits += 1

    return hits / len(questions)

In [25]:
def mrr(questions, all_results):
    total = 0.0

    for q in questions:
        retrieved = all_results[q["question_id"]]

        if q["gold_passage_id"] in retrieved:
            rank = retrieved.index(q["gold_passage_id"]) + 1
            total += 1.0 / rank

    return total / len(questions)

In [26]:
for k in [1, 3, 5, 10]:
    print(
        f"Dense Recall@{k}:",
        recall_at_k(questions, dense_results, k)
    )

print("Dense MRR:", mrr(questions, dense_results))

Dense Recall@1: 0.76
Dense Recall@3: 0.95
Dense Recall@5: 0.95
Dense Recall@10: 0.97
Dense MRR: 0.846111111111111


In [29]:
import json

with open("results/bm25_top10.json", encoding="utf-8") as f:
    all_results = json.load(f)

In [30]:
print(f"{'k':<5}{'BM25 Recall':<15}{'Dense Recall':<15}")

for k in [1, 3, 5, 10]:
    b = recall_at_k(questions, all_results, k)
    d = recall_at_k(questions, dense_results, k)

    print(f"{k:<5}{b:<15.3f}{d:<15.3f}")

k    BM25 Recall    Dense Recall   
1    0.790          0.760          
3    0.880          0.950          
5    0.910          0.950          
10   0.950          0.970          


In [31]:
print("BM25 MRR:", mrr(questions, all_results))
print("Dense MRR:", mrr(questions, dense_results))

BM25 MRR: 0.8459682539682539
Dense MRR: 0.846111111111111


In [32]:
bm25_wins = []
dense_wins = []
both_win = []
both_lose = []

for q in questions:

    qid = q["question_id"]

    b_hit = q["gold_passage_id"] in all_results[qid][:5]
    d_hit = q["gold_passage_id"] in dense_results[qid][:5]

    if b_hit and not d_hit:
        bm25_wins.append(q)

    elif d_hit and not b_hit:
        dense_wins.append(q)

    elif b_hit and d_hit:
        both_win.append(q)

    else:
        both_lose.append(q)

print("BM25 only:", len(bm25_wins))
print("Dense only:", len(dense_wins))
print("Both:", len(both_win))
print("Neither:", len(both_lose))

BM25 only: 2
Dense only: 6
Both: 89
Neither: 3


In [33]:
for q in dense_wins[:3]:

    print("Question:", q["question"])

    print(
        "Gold passage:",
        corpus[q["gold_passage_id"]]["text"][:300]
    )

    print("-" * 50)

Question: Why would a teacher's college exist?
Gold passage: There are a variety of bodies designed to instill, preserve and update the knowledge and professional standing of teachers. Around the world many governments operate teacher's colleges, which are generally established to serve and protect the public interest through certifying, governing and enforci
--------------------------------------------------
Question: What is th elast name of the player who was the Super Bowl 50 winner's leading rusher?
Gold passage: Manning finished the game 13 of 23 for 141 yards with one interception and zero touchdowns. Sanders was his top receiver with six receptions for 83 yards. Anderson was the game's leading rusher with 90 yards and a touchdown, along with four receptions for 10 yards. Miller had six total tackles (five
--------------------------------------------------
Question: What work is useful for pastors?
Gold passage: Luther's Small Catechism proved especially effective in helping par

In [34]:
for q in bm25_wins[:3]:

    print("Question:", q["question"])

    print(
        "Gold passage:",
        corpus[q["gold_passage_id"]]["text"][:300]
    )

    print("-" * 50)

Question: What profession does Zbigniew Badowski have?
Gold passage: Another important library – the University Library, founded in 1816, is home to over two million items. The building was designed by architects Marek Budzyński and Zbigniew Badowski and opened on 15 December 1999. It is surrounded by green. The University Library garden, designed by Irena Bajerska, 
--------------------------------------------------
Question: What causes the population of ctenophora to grow at an explosive rate?
Gold passage: Most species are hermaphrodites—a single animal can produce both eggs and sperm, meaning it can fertilize its own egg, not needing a mate. Some are simultaneous hermaphrodites, which can produce both eggs and sperm at the same time. Others are sequential hermaphrodites, in which the eggs and sperm m
--------------------------------------------------


In [35]:
import csv

with open(
    "results/dense_experiments.csv",
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "run_name",
        "model_name",
        "recall@1",
        "recall@3",
        "recall@5",
        "recall@10",
        "mrr"
    ])

    writer.writerow([
        "dense_baseline",
        "intfloat/multilingual-e5-base",
        recall_at_k(questions, dense_results, 1),
        recall_at_k(questions, dense_results, 3),
        recall_at_k(questions, dense_results, 5),
        recall_at_k(questions, dense_results, 10),
        mrr(questions, dense_results)
    ])

In [37]:
with open(
    "results/dense_top10.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        dense_results,
        f,
        indent=2
    )